In [ ]:
# Imports
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gc # Garbage Collector
import os
import sys

# --- IMPORTS DEEP LEARNING ---
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Embedding, Flatten, concatenate, Dropout
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import mean_squared_log_error
import re

In [ ]:
data = pd.read_csv('data/processed_data.csv')
data.info()

In [ ]:
# --- LOG-TRANSFORMATION DE LA CIBLE ---

# On applique la transformation seulement sur les donn?es d'entra?nement (is_train == 1)
# car la colonne 'sales' du jeu de test contient des np.nan.

data.loc[data['is_train'] == 1, 'sales'] = np.log1p(data.loc[data['is_train'] == 1, 'sales'])

In [ ]:
# ==============================================================================
# ?TAPE 1: PR?PARATION FINALE POUR LE MOD?LE DEEP LEARNING
# ==============================================================================
print("Pr?paration finale des donn?es pour MLP...")

# 2. Nettoyage des noms de colonnes (pour LightGBM, mais bonne pratique aussi pour Keras)
def clean_feature_names(df):
    new_cols = []
    for col in df.columns:
        col = re.sub(r'[\[\]<>]', '', col)
        col = re.sub(r'\s+', '_', col)
        col = re.sub(r'[,()"-]', '', col)
        new_cols.append(col)
    df.columns = new_cols
    return df
data = clean_feature_names(data)

# 3. Supprimer la colonne transactions originale qui a servi ? cr?er les lags
if 'transactions' in data.columns:
    data.drop(columns=['transactions'], inplace=True)

# D?finition des features
FEATURES = [col for col in data.columns if col not in ['id', 'sales', 'is_train', 'date']]

# S?paration des features en types pour le mod?le DL
CATEGORICAL_FEATURES_OHE = ['is_holiday', 'dayofweek', 'is_weekend', 'is_payday'] # Simple OHE (ou binaire)
CATEGORICAL_FEATURES_EMBEDDING = ['store_nbr', 'family', 'city', 'state', 'type', 'cluster', 'month', 'weekofyear'] # Mieux adapt? aux Embeddings
NUMERICAL_FEATURES = [col for col in FEATURES if col not in CATEGORICAL_FEATURES_OHE and col not in CATEGORICAL_FEATURES_EMBEDDING]

# --- Op?rations de Pr?traitement Finales sur Dataframe ---

# 4. Encodage Label (Obligatoire pour les Embeddings)
# Les colonnes d'Embedding DOIVENT commencer ? 0.
label_encoders = {}
for col in CATEGORICAL_FEATURES_EMBEDDING:
    if col in data.columns:
        le = LabelEncoder()
        # Le fit sur l'ensemble complet (train + test) pour ?viter les cl?s inconnues
        data[col] = le.fit_transform(data[col])
        label_encoders[col] = le

# 5. Normalisation des Features Num?riques (MinMaxScaler)
scaler = MinMaxScaler()
data[NUMERICAL_FEATURES] = scaler.fit_transform(data[NUMERICAL_FEATURES])

# 6. One-Hot Encoding pour les petites cat?gories
data = pd.get_dummies(data, columns=CATEGORICAL_FEATURES_OHE, drop_first=True)

# Mise ? jour des FEATURES (si de nouvelles colonnes OHE ont ?t? cr??es)
FEATURES = [col for col in data.columns if col not in ['id', 'sales', 'is_train', 'date']]

In [ ]:
# ==============================================================================
# ?TAPE 2: S?PARATION DES DONN?ES EN ENTRA?NEMENT ET TEST
# ==============================================================================
print("S?paration des jeux d'entra?nement et de test...")
train_df = data[data['is_train'] == 1].copy()
test_df = data[data['is_train'] == 0].copy()

In [ ]:
# ==============================================================================
# ?TAPE 3 BIS: CR?ATION D'UN JEU DE VALIDATION
# ==============================================================================
print("Avant conversion, dtype date:", train_df['date'].dtype)

# Forcer 'date' en datetime
train_df['date'] = pd.to_datetime(train_df['date'])

print("Apr?s conversion, dtype date:", train_df['date'].dtype)

print("Cr?ation du jeu de validation...")

# On utilise la colonne 'date' QUI EXISTE D?J? DANS train_df
last_train_date = train_df['date'].max()
validation_start_date = last_train_date - pd.DateOffset(days=15)

# On cr?e le masque DIRECTEMENT ? partir de train_df. Les longueurs correspondront parfaitement.
valid_indices = train_df[train_df['date'] >= validation_start_date].index
train_indices = train_df[train_df['date'] < validation_start_date].index

# Cr?er les jeux de donn?es partiels
train_part_df = train_df.loc[train_indices]
valid_df = train_df.loc[valid_indices]

print(f"Jeu d'entra?nement partiel : {train_part_df.shape[0]} lignes")
print(f"Jeu de validation : {valid_df.shape[0]} lignes")

In [ ]:
# ==============================================================================
# ?TAPE 4: D?FINITION DES FEATURES (X) ET DE LA CIBLE (y)
# ==============================================================================

# S?paration des jeux de donn?es apr?s le nettoyage et l'encodage
train_df = data[data['is_train'] == 1].copy()
test_df = data[data['is_train'] == 0].copy()

# 1. S?paration Train Part/Validation
train_part_df = train_df.loc[train_indices].copy()
valid_df = train_df.loc[valid_indices].copy()

# Cible y (d?j? log-transform?e dans la nouvelle ?tape 1)
y_train_part = train_part_df['sales'].values
y_valid = valid_df['sales'].values

# Features X
X_train_part_num = train_part_df[NUMERICAL_FEATURES].values
X_valid_num = valid_df[NUMERICAL_FEATURES].values
X_test_num = test_df[NUMERICAL_FEATURES].values

# Features OHE/Embedding (Attention ? l'ordre des colonnes, Keras est strict)
X_train_part_cat = train_part_df[[col for col in FEATURES if col not in NUMERICAL_FEATURES]].values
X_valid_cat = valid_df[[col for col in FEATURES if col not in NUMERICAL_FEATURES]].values
X_test_cat = test_df[[col for col in FEATURES if col not in NUMERICAL_FEATURES]].values

test_ids = test_df['id'].copy()

# Lib?rer de la m?moire
del train_df, test_df, data, train_part_df, valid_df
gc.collect()

In [ ]:
# Debug rapide des dtypes
print("X_train_part_num dtypes:", X_train_part_num.dtype)
print("X_train_part_cat dtypes:", X_train_part_cat.dtype)
print("y_train_part dtype:", y_train_part.dtype)

# Conversion en types compatibles Keras
X_train_part_num = X_train_part_num.astype("float32")
X_valid_num = X_valid_num.astype("float32")
X_test_num = X_test_num.astype("float32")

X_train_part_cat = X_train_part_cat.astype("int32")
X_valid_cat = X_valid_cat.astype("int32")
X_test_cat = X_test_cat.astype("int32")

y_train_part = y_train_part.astype("float32")
y_valid = y_valid.astype("float32")

In [ ]:
# ==============================================================================
# ?TAPE 5: ENTRA?NEMENT DU MOD?LE MLP (Deep Learning)
# ==============================================================================
print("\nEntra?nement du Mod?le MLP Global...")

# --- 5.1: D?finition de l'Architecture ---

def create_mlp_model(numerical_features, embedding_features, categorical_features_ohe, label_encoders):
    # Dictionnaire pour stocker les inputs Keras (une par feature)
    inputs = {}
    
    # 1. Inputs pour les Embeddings (features cat?gorielles avec bcp de classes)
    embedding_outputs = []
    for col in embedding_features:
        # Input: Entiers (Label Encoded)
        vocab_size = len(label_encoders[col].classes_)
        embed_dim = max(1, vocab_size // 4) # Taille de l'embedding: sqrt(vocab_size) ou vocab_size / 4
        
        input_layer = Input(shape=(1,), name=f'input_{col}')
        inputs[col] = input_layer
        
        # Couche d'Embedding
        embedding = Embedding(input_dim=vocab_size, output_dim=embed_dim, name=f'embedding_{col}')(input_layer)
        flatten = Flatten()(embedding)
        embedding_outputs.append(flatten)
        
    # 2. Inputs pour les features OHE (d?j? g?r?es par pd.get_dummies) et Num?riques
    
    # Keras ne prend qu'un seul grand input pour les features num?riques/OHE
    input_numeric_ohe = Input(shape=(X_train_part_num.shape[1] + (X_train_part_cat.shape[1] - len(embedding_features)),), 
                          name='input_num_ohe')
    inputs['num_ohe'] = input_numeric_ohe
    
    # 3. Concatenation
    if embedding_outputs:
        # Concat?ner les Embeddings avec les features num?riques/OHE
        all_features = concatenate(embedding_outputs + [input_numeric_ohe])
    else:
        all_features = input_numeric_ohe
        
    # 4. Couches Dense du MLP
    x = Dense(1024, activation='relu')(all_features)
    x = Dropout(0.3)(x)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.3)(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.2)(x)
    
    # Output: 1 neurone pour la vente (r?gression)
    output_layer = Dense(1, activation='linear', name='output_sales')(x)
    
    # Construction et compilation du mod?le
    # Le mod?le prend toutes les couches d'entr?e d?finies
    model = Model(inputs=list(inputs.values()), outputs=output_layer)
    model.compile(optimizer='adam', loss='mse', metrics=[tf.keras.metrics.RootMeanSquaredError()])
    return model

# --- 5.2: Cr?ation et Entra?nement ---

# Pr?parer les inputs pour Keras (un input par embedding + l'input num/ohe)
def prepare_keras_inputs(X_cat, X_num, embedding_features):
    # X_cat = [embeddings | autres cat/OHE]
    
    X_embed = X_cat[:, :len(embedding_features)]
    X_ohe = X_cat[:, len(embedding_features):]
    
    keras_inputs = []
    
    # 1. Inputs d'Embedding (s?par?s, 1 colonne ? la fois)
    for i in range(len(embedding_features)):
        # Keras veut un array de shape (N, 1) pour l'input d'Embedding
        keras_inputs.append(X_embed[:, i].reshape(-1, 1))
        
    # 2. Input Num?rique/OHE (fusionn?)
    X_num_ohe = np.hstack([X_num, X_ohe])
    keras_inputs.append(X_num_ohe)
    
    return keras_inputs

# Cr?ation du mod?le
mlp_model = create_mlp_model(NUMERICAL_FEATURES, CATEGORICAL_FEATURES_EMBEDDING, CATEGORICAL_FEATURES_OHE, label_encoders)
# mlp_model.summary() # D?commenter pour voir le r?sum? du mod?le

# Pr?paration des inputs
X_train_keras = prepare_keras_inputs(X_train_part_cat, X_train_part_num, CATEGORICAL_FEATURES_EMBEDDING)
X_valid_keras = prepare_keras_inputs(X_valid_cat, X_valid_num, CATEGORICAL_FEATURES_EMBEDDING)

# Entra?nement
history = mlp_model.fit(
    X_train_keras, y_train_part,
    validation_data=(X_valid_keras, y_valid),
    epochs=10, # Commence avec 50, ajuster en fonction du temps et de la performance
    batch_size=4096, # Taille de lot ?lev?e pour acc?l?rer le training
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)]
)

In [ ]:
# ==============================================================================
# ?TAPE 6: PR?DICTION ET CR?ATION DU FICHIER DE SOUMISSION
# ==============================================================================
print("\nG?n?ration des pr?dictions sur le jeu de test...")

# Pr?paration de l'input de test
X_test_keras = prepare_keras_inputs(X_test_cat, X_test_num, CATEGORICAL_FEATURES_EMBEDDING)

# Pr?diction (sortie log-transform?e)
predictions_log = mlp_model.predict(X_test_keras).flatten()

# Inverse de la Log Transformation (np.expm1)
predictions = np.expm1(predictions_log)
predictions[predictions < 0] = 0

submission_df = pd.DataFrame({'id': test_ids, 'sales': predictions})
submission_df.to_csv('submission_mlp_deeplearning.csv', index=False)

print("\nFichier 'submission_mlp_deeplearning.csv' cr?? avec succ?s !")
print(submission_df.head())